# Grid Search CV

In [ ]:
import numpy as np
import pandas as pd

from sktime.forecasting.arch import ARCH
# Modules for cross-validation
from sktime.split import temporal_train_test_split, ExpandingWindowSplitter, SlidingWindowSplitter, SingleWindowSplitter
from sktime.performance_metrics.forecasting import MeanSquaredError, MeanAbsoluteError, MeanAbsolutePercentageError # MSE, MAE, MAPE metrics
# Grid search for optimal hyperparameters
from sktime.forecasting.model_selection import ForecastingGridSearchCV

import pandas_datareader.data as web

# visualization settings
import matplotlib.pyplot as plt

# Do not show warnings
import warnings
warnings.simplefilter(action='ignore', category=Warning)

Load monthly data from the [`FRED`](https://fred.stlouisfed.org/) database on Market Yield on U.S. Treasury Securities at 10-Year Constant Maturity (Symbol [`GS10`](https://fred.stlouisfed.org/series/GS10)) from 2000-01-01 to the present and create dataframe `y` - first difference

In [ ]:
y = web.DataReader(name='GS10', data_source='fred', start='2000-01').diff().dropna()
y.index = y.index.to_period(freq='M')

Search for optimal model (hyper)parameters by grid

In [ ]:
# Define the forecasting method
forecaster = ARCH(mean='AR')

# set cross-validation parameters
cv_strategy = ExpandingWindowSplitter(fh=np.arange(1, 6), initial_window=100, step_length=5)
# cv_strategy = SlidingWindowSplitter(fh=np.arange(1, 6), initial_window=100, step_length=5)

# Define the grid of model parameter values as a dictionary
# Important: all possible combinations will be enumerated!
param_grid = {'lags':[0, 1, 2], 'p': [0, 1, 2], 'o': [0, 1, 2], 'q': [0, 1, 2], 'vol':['GARCH', 'EGARCH'], 'dist':['normal', 't', 'skewt']}

# initialize the metric
metric = MeanSquaredError(square_root=False)
# metric = MeanAbsoluteError()
# metric = MeanAbsolutePercentageError()

# Grid search
gscv = ForecastingGridSearchCV(forecaster=forecaster, param_grid=param_grid, cv=cv_strategy, scoring=metric)

gscv.fit(y)

In [ ]:
# Parameters of the optimal model
gscv.get_fitted_params()['best_forecaster']